# Job ETL da silver

Neste notebook, é aplicado o Job ETL. Ele é um processo que extrai, transforma e carrega dados de diferentes fontes para um destino central. No caso desse projeto, a fonte será extraida da camada silver para a gold pelo arquivo Complete_Pokedex-Tratada.csv e o resultado será armazenado em outro csv e utilizado na camada gold.

### Frameworks utilizados

In [6]:
import pandas as pd
import psycopg2
import time
import sqlalchemy
import warnings

### Extrair

In [7]:
# Suppress the pandas warning
warnings.filterwarnings('ignore', category=UserWarning)

def get_db_connection():
    while True:
        try:
            conexao = psycopg2.connect(
                host="localhost",
                port=5432,
                database="pokedex_db",
                user="pokedex_user",
                password="pokedex_password"
            )
            return conexao
        except psycopg2.OperationalError:
            print("O banco não está pronto, aguardando 3 segundos...")
            time.sleep(3)

# Get data from database instead of CSV
print("Conectando ao banco de dados...")
conexao = get_db_connection()
df = pd.read_sql_query("SELECT * FROM pokemon", conexao)
conexao.close()

print("Dados carregados do banco de dados:")
print(df.head())
print(f"Total de registros: {len(df)}")

Conectando ao banco de dados...
Dados carregados do banco de dados:
   pokedex_number   pokemon_name type_1  type_2  height  weight  hit_points  \
0               1      Bulbasaur  Grass  Poison     0.7     6.9          45   
1               2        Ivysaur  Grass  Poison     1.0    13.0          60   
2               3  Mega Venusaur  Grass  Poison     2.4   155.5          80   
3               4     Charmander   Fire             0.6     8.5          39   
4               5     Charmeleon   Fire             1.1    19.0          58   

   attack  defense  total_stats  ...  against_ground  against_flying  \
0      49       49          318  ...             1.0             2.0   
1      62       63          405  ...             1.0             2.0   
2     100      123          625  ...             1.0             2.0   
3      52       43          309  ...             2.0             1.0   
4      64       58          405  ...             2.0             1.0   

   against_psychic  agai

### Transformar


In [8]:
# Apaga colunas
colunas_para_apagar = [
'hit_points',
'base_happiness',
'evolves_from',
'mythical',
'genderless', 
'female_rate', 
'egg_cycles'
]

df_tratado = df.drop(columns=colunas_para_apagar)

# alterando nome do id 
df_tratado = df.rename(columns={'pokedex_number': 'SRK_pok'})


print(df_tratado.head())

print("\n transformação concluída!")

   SRK_pok   pokemon_name type_1  type_2  height  weight  hit_points  attack  \
0        1      Bulbasaur  Grass  Poison     0.7     6.9          45      49   
1        2        Ivysaur  Grass  Poison     1.0    13.0          60      62   
2        3  Mega Venusaur  Grass  Poison     2.4   155.5          80     100   
3        4     Charmander   Fire             0.6     8.5          39      52   
4        5     Charmeleon   Fire             1.1    19.0          58      64   

   defense  total_stats  ...  against_ground  against_flying  against_psychic  \
0       49          318  ...             1.0             2.0              2.0   
1       63          405  ...             1.0             2.0              2.0   
2      123          625  ...             1.0             2.0              2.0   
3       43          309  ...             2.0             1.0              1.0   
4       58          405  ...             2.0             1.0              1.0   

   against_bug against_rock agai

### Carregar 

##### salva dado tratado carregando em um novo csv

In [10]:
# Load data to database (Gold layer)
conexao = get_db_connection()
cursor = conexao.cursor()

# Create Dim_pokmn (POKEMON)
cursor.execute("""
CREATE TABLE IF NOT EXISTS Dim_pkm (
    SRK_pkn SERIAL PRIMARY KEY,
    pkm_nam VARCHAR(50) NOT NULL,
    tp1 VARCHAR(50) NOT NULL,
    tp2 VARCHAR(50),
    hgt DOUBLE PRECISION NOT NULL,
    wgt DOUBLE PRECISION NOT NULL,
    gen INT NOT NULL,
    leg BOOLEAN NOT NULL,
    mga_evl BOOLEAN NOT NULL,
    all_frm BOOLEAN NOT NULL,
    glr_frm BOOLEAN NOT NULL,
    swt_frm BOOLEAN NOT NULL
);
""")

# Create Dim_batlh (BATALHA)
cursor.execute("""
CREATE TABLE IF NOT EXISTS Dim_btl (
    SRK_btl SERIAL PRIMARY KEY,
    atk INT NOT NULL,
    dfs INT NOT NULL,
    cap_rte INT NOT NULL,
    bas_exp INT NOT NULL,
    exp_tpe VARCHAR(50) NOT NULL
);
""")

# Create Dim_efetContr (EFETIVIDADE CONTRA)
cursor.execute("""
CREATE TABLE IF NOT EXISTS Dim_efetContr (
    SRK_eft SERIAL PRIMARY KEY,
    agt_nrm DOUBLE PRECISION NOT NULL,
    agt_fre DOUBLE PRECISION NOT NULL,
    agt_wtr DOUBLE PRECISION NOT NULL,
    agt_elt DOUBLE PRECISION NOT NULL,
    agt_grs DOUBLE PRECISION NOT NULL,
    agt_ice DOUBLE PRECISION NOT NULL,
    agt_fgt DOUBLE PRECISION NOT NULL,
    agt_psn DOUBLE PRECISION NOT NULL,
    agt_gnd DOUBLE PRECISION NOT NULL,
    agt_fly DOUBLE PRECISION NOT NULL,
    agt_psy DOUBLE PRECISION NOT NULL,
    agt_bug DOUBLE PRECISION NOT NULL,
    agt_rck DOUBLE PRECISION NOT NULL,
    agt_gst DOUBLE PRECISION NOT NULL,
    agt_drg DOUBLE PRECISION NOT NULL,
    agt_drk DOUBLE PRECISION NOT NULL,
    agt_stl DOUBLE PRECISION NOT NULL,
    agt_fry DOUBLE PRECISION NOT NULL
);
""")

# Create Fat_pokdx (POKEDEX)
cursor.execute("""
CREATE TABLE IF NOT EXISTS Fat_pokdx (
    SRK_pkx SERIAL PRIMARY KEY,
    SRK_pkn INT NOT NULL,
    SRK_btl INT NOT NULL,
    SRK_eft INT NOT NULL,
    
    -- Foreign Key Constraints
    CONSTRAINT fk_pokemon
        FOREIGN KEY (SRK_pkn)
        REFERENCES Dim_pkm (SRK_pkn),
        
    CONSTRAINT fk_batalha
        FOREIGN KEY (SRK_btl)
        REFERENCES Dim_btl (SRK_btl),
        
    CONSTRAINT fk_efetividade
        FOREIGN KEY (SRK_eft)
        REFERENCES Dim_efetContr (SRK_eft)
);
""")

##### popula dados no banco

In [12]:
# Insert data into Gold table
for index, row in df_tratado.iterrows():
    cursor.execute("""
    INSERT INTO pokemon_gold (
        SRK_pok, pkm_nam, tp1, tp2, hgt, wgt, 
        atk, dfs, cap_rte, gen, bas_exp, 
        exp_tpe, mga_evl, all_frm, glr_frm, 
        swt_frm, leg, agt_nrm, agt_fre, 
        agt_wtr, agt_elt, agt_grs, agt_ice, 
        agt_fgt, agt_psn, agt_gnd, agt_fly, 
        agt_psy, agt_bug, agt_rck, agt_gst, 
        agt_drg, agt_drk, agt_stl, agt_fry
    ) VALUES (%d, %s, %s, %s, %f, %f, %d, %d, %d, %d, %d, %s, %s, %s, %s, %s, %s, %f, %f, %f, %f, %f, %f, %f, %f, %f, %f, %f, %f, %f, %f, %f, %f, %f, %f)
    """, tuple(row))

conexao.commit()

# Fechando conexão
cursor.close()
conexao.close()

print("ETL Gold concluído! Dados carregados na tabela pokemon_gold.")

ValueError: unsupported format character 'd' (0x64) at index 400